# Model Comparison — MLX (Apple Silicon)

Runs entirely on-device via `mlx-lm`. Evaluates 3 base models + 3 LoRA-adapted variants across 38 symbols (2022–2025).

**One-time setup:** the conversion cells below merge LoRA adapters into their base models and convert everything to quantized MLX format. After that, inference is fast and fully offline.

### Installations

In [ ]:
%pip install mlx-lm transformers peft safetensors huggingface_hub -q
%pip install plotly pandas psutil -q
%pip install "nbformat>=4.2.0" -q

In [ ]:
# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Secrets

In [ ]:
HF_TOKEN=''
ALPACA_API_KEY=''
ALPACA_SECRET_KEY=''

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

### Configuration

In [ ]:
MODEL_CONFIGS = [
    # {
    #     'label':       'Qwen2.5-7B base',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    # {
    #     'label':       'Qwen2.5-7B LoRA v1-500',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit',
    #     'mlx_adapter': 'mlx_adapters/qwen7b_lora',
    #     'base_label':  'Qwen2.5-7B base',
    # },
    # {
    #     'label':       'Llama-3.1-8B base',
    #     'mlx_id':      'mlx-community/Meta-Llama-3.1-8B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    # {
    #     'label':       'Llama-3.1-8B LoRA v1-500',
    #     'mlx_id':      'mlx-community/Meta-Llama-3.1-8B-Instruct-4bit',
    #     'mlx_adapter': 'mlx_adapters/llama8b_lora',
    #     'base_label':  'Llama-3.1-8B base',
    # },
    # {
    #     'label':       'Qwen2.5-32B base',
    #     'mlx_id':      'mlx-community/Qwen2.5-Coder-32B-Instruct-4bit',
    #     'mlx_adapter': None,
    # },
    {
        'label':       'Qwen2.5-32B LoRA v1-500',
        'mlx_id':      'mlx-community/Qwen2.5-Coder-32B-Instruct-4bit',
        'mlx_adapter': 'mlx_adapters/qwen32b_lora',
        'base_label':  'Qwen2.5-32B base',
    },
]

# TEST_START = '2022-01-01'
# TEST_END   = '2025-12-31'
N_SAMPLES  = 5


# from trading_rl import TRAINING_SYMBOLS as SYMBOLS  # full 38-symbol universe

SYMBOLS = ['AAPL', 'AMZN', 'MSFT', 'TSLA', 'GOOGL'] # BENCHMARK_SYMBOLS
TEST_START   = '2022-06-01' # BENCHMARK_START
TEST_END     = '2024-01-01' # BENCHMARK_END


### One-Time Model Conversion

Converts each model to 4-bit quantized MLX format and saves locally.
Subsequent runs skip configs whose `mlx_path` already exists.

> **32B LoRA note:** merging a 32B peft adapter requires ~64 GB RAM (float16 weights).
> If you have 32 GB, only the base 32B model can be converted here.
> To get the 32B LoRA: merge on Colab/Kaggle A100, download the merged MLX folder, place it at `mlx_models/qwen32b_lora`.

In [ ]:
# One-time setup: run convert_lora_adapters.ipynb to create mlx_adapters/ before evaluating LoRA configs.

In [ ]:
import os
for cfg in MODEL_CONFIGS:
    adapter = cfg.get('mlx_adapter')
    if adapter:
        status = 'ready' if os.path.exists(adapter) else 'MISSING — run convert_lora_adapters.ipynb'
        print(f"  {cfg['label']}: adapter {adapter} [{status}]")
    else:
        print(f"  {cfg['label']}: base model ({cfg['mlx_id']})")

### Core Classes

In [ ]:
import os
import gc
import json

import pandas as pd
import plotly.express as px
from IPython.display import display
from mlx_lm import load as mlx_load, generate as mlx_gen

from trading_rl import Backtester, evaluate_symbol, make_prompt

In [ ]:
from mlx_lm.sample_utils import make_sampler

class MLXModel:
    def __init__(self, mlx_path, adapter_path=None, max_tokens=512, temp=1.0):
        self.max_tokens = max_tokens
        self.sampler = make_sampler(temp=temp)
        print(f'Loading {mlx_path} ...')
        self.model, self.tokenizer = mlx_load(mlx_path, adapter_path=adapter_path)
        if adapter_path:
            print(f'Adapter: {adapter_path}')
        print('Ready.')

    def generate(self, inputs):
        if hasattr(self.tokenizer, 'apply_chat_template'):
            prompt = self.tokenizer.apply_chat_template(
                [{'role': 'user', 'content': inputs.strip()}],
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = inputs.strip()
        return mlx_gen(
            self.model, self.tokenizer,
            prompt=prompt,
            max_tokens=self.max_tokens,
            sampler=self.sampler,
            verbose=False,
        )

    def unload(self):
        del self.model, self.tokenizer
        gc.collect()
        print('Model unloaded.')

### Evaluation Functions

In [ ]:
def evaluate_model(config, bt_instance, symbols=None, start=TEST_START, end=TEST_END, n_samples=N_SAMPLES):
    symbols      = symbols or SYMBOLS
    label        = config['label']
    mlx_id       = config['mlx_id']
    adapter_path = config.get('mlx_adapter')

    if adapter_path and not os.path.exists(adapter_path):
        print(f'SKIP {label}: adapter {adapter_path} not found. Run convert_lora_adapters.ipynb first.')
        return []

    print(f"\n{'='*60}")
    print(f'Evaluating: {label}')
    print(f'Model:      {mlx_id}')
    if adapter_path:
        print(f'Adapter:    {adapter_path}')
    print(f'Symbols: {len(symbols)}  |  Samples/symbol: {n_samples}  |  {start} to {end}')
    print(f"{'='*60}")

    model   = MLXModel(mlx_id, adapter_path=adapter_path)
    records = []
    for sym in symbols:
        sym_recs = evaluate_symbol(model.generate, bt_instance, sym, start, end, n_samples,
                                   prompt=make_prompt(sym, start, end, compact=True))
        for r in sym_recs:
            r['model'] = label
        records.extend(sym_recs)

    model.unload()
    ok = sum(1 for r in records if r['status'] == 'profitable')
    print(f'[{label}] Done. Profitable: {ok}/{len(records)}')
    return records

### Run Evaluation

In [ ]:
bt_instance = Backtester(ALPACA_API_KEY, ALPACA_SECRET_KEY, verbose=False)
bt_instance.load_bars(SYMBOLS, TEST_START, TEST_END)

In [ ]:
all_results = []
for config in MODEL_CONFIGS:
    records = evaluate_model(config, bt_instance)
    all_results.extend(records)

df = pd.DataFrame(all_results)
print(f'\nTotal samples collected: {len(df)}')
df.head(10)

In [ ]:
df = pd.DataFrame(all_results)
print(f'\nTotal samples collected: {len(df)}')
df.head(10)

In [ ]:
df.to_csv('model_comparison_results_2.csv', index=False)
print('Saved to model_comparison_results_2.csv')

### Results Analysis

In [ ]:
summary = df.groupby('model').agg(
    total=('sample', 'count'),
    valid=('status', lambda x: x.isin(['profitable', 'loss']).sum()),
    profitable=('status', lambda x: (x == 'profitable').sum()),
    loss=('status', lambda x: (x == 'loss').sum()),
    no_trades=('status', lambda x: (x == 'no_trades').sum()),
    invalid=('status', lambda x: x.isin(['missing_methods', 'invalid_code', 'exception', 'timeout']).sum()),
    
    mean_reward=('reward_score', 'mean'),
    mean_return=('return_pct', 'mean'),
    mean_sharpe=('sharpe_ratio', 'mean'),
    mean_annual=('avg_annual_return_pct', 'mean'),
).reset_index()
summary['profitable_pct'] = (summary['profitable'] / summary['total'] * 100).round(1)
summary['valid_pct'] = (summary['valid'] / summary['total'] * 100).round(1)
display(summary.round(3))

In [ ]:
status_counts = df.groupby(['model', 'status']).size().reset_index(name='count')
fig = px.bar(
    status_counts, x='model', y='count', color='status', barmode='stack',
    title='Outcome Breakdown by Model',
    color_discrete_map={
        'profitable': '#2ecc71', 'loss': '#e67e22', 'no_trades': '#95a5a6',
        'missing_methods': '#e74c3c', 'invalid_code': '#c0392b',
        'exception': '#9b59b6', 'timeout': '#7f8c8d',
    },
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=30)
fig.show()

In [ ]:
best = (
    df[df['reward_score'].notna()]
    .sort_values('reward_score', ascending=False)
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
)

pivot = best.pivot_table(index='symbol', columns='model', values='reward_score')
melted = pivot.reset_index().melt(id_vars='symbol', var_name='model', value_name='best_reward')
fig = px.bar(
    melted, x='symbol', y='best_reward', color='model', barmode='group',
    title='Best Reward Score per Symbol — Model Comparison',
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show()

In [ ]:
pivot_ret = best.pivot_table(index='symbol', columns='model', values='return_pct')
for config in MODEL_CONFIGS:
    if not config.get('hf_adapter') or not config.get('base_label'):
        continue
    adapted_label = config['label']
    base_label    = config['base_label']
    if base_label not in pivot_ret.columns or adapted_label not in pivot_ret.columns:
        continue
    delta = (pivot_ret[adapted_label] - pivot_ret[base_label]).to_frame(name='delta_return_pct')
    fig = px.imshow(
        delta.T,
        title=f'Return Delta: {adapted_label} vs {base_label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
save_dir = 'model_comparison_strategies'
os.makedirs(save_dir, exist_ok=True)
saved = 0

for _, row in best[best['status'] == 'profitable'].iterrows():
    model_dir = os.path.join(save_dir, row['model'].replace('/', '_').replace(' ', '_'))
    os.makedirs(model_dir, exist_ok=True)

    code = row.get('strategy_code')
    if code:
        with open(os.path.join(model_dir, f"{row['symbol']}_strategy.py"), 'w') as f:
            f.write(code)

    stats = {
        'model':                  row['model'],
        'symbol':                 row['symbol'],
        'return_pct':             float(row['return_pct'])             if pd.notna(row['return_pct'])             else None,
        'sharpe_ratio':           float(row['sharpe_ratio'])           if pd.notna(row['sharpe_ratio'])           else None,
        'avg_annual_return_pct':  float(row['avg_annual_return_pct'])  if pd.notna(row['avg_annual_return_pct'])  else None,
        'max_drawdown_pct':       float(row['max_drawdown_pct'])       if pd.notna(row['max_drawdown_pct'])       else None,
    }
    with open(os.path.join(model_dir, f"{row['symbol']}_stats.json"), 'w') as fj:
        json.dump(stats, fj, indent=2)
    saved += 1

print(f'Saved {saved} profitable strategies to {save_dir}/')